# DTAT361. deeptrack.sources.base

<a href="https://colab.research.google.com/github/DeepTrackAI/DeepTrack2/blob/develop/tutorials/3-advanced-topics/DTAT361.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# !pip install deeptrack  # Uncomment if running on Colab/Kaggle.

This advanced tutorial introduces the module [deeptrack.sources.base](../../deeptrack/sources/base.py).

## 1. What is `base.py`?

The `base.py` module provides utilities for manipulating data sources, primarily when data needs to be dynamically manipulated, filtered, or combined. This guide explains how to use each component in the module with examples.

### The key roles of `base.py` are:

- **Data Wrapping and Activation:**
  `base.py` introduces the `Source` and `SourceItem` classes to represent structured data streams. Each `SourceItem` triggers a list of callbacks when accessed, enabling dynamic updates and evaluation within DeepTrack2 pipelines.

- **Field Access via Nodes:**
  The module defines `SourceDeepTrackNode`, a subclass of `DeepTrackNode`, which allows dynamic field access using attribute notation. This makes it possible to write expressions like `source.a()` that automatically track dependencies on the current index.

- **Source Manipulation Utilities:**
  It provides tools such as `Product`, `Subset`, `Sources`, and `random_split()` to construct complex data sources by combining, filtering, and partitioning existing ones. These tools allow efficient data manipulation for training/validation/test workflows and augmentation pipelines.

## 2.  Dynamically Generating Child Nodes with `SourceDeepTrackNode`

The `SourceDeepTrackNode` class extends `DeepTrackNode` by enabling structured, dynamic access to dictionary-like data. When you access an attribute of a `SourceDeepTrackNode`, it automatically generates a child node corresponding to that key. This is particularly useful when you want to track dependencies between data fields and features (such as `source.position.x` or `source.a()`) in a dynamic way.

This node-based approach supports:
- Automatic dependency tracking in computation graphs.
- Composable, lazy evaluation of nested or hierarchical data.
- Dynamic binding to the currently activated data item.

### 2.1. Accessing Top-Level Fields

Define parent node with `SourceDeepTrackNode`.

In [2]:
from deeptrack.sources.base import SourceDeepTrackNode

node = SourceDeepTrackNode(lambda: {"a": 10, "b": 20})

Dynamically generate child nodes.

In [3]:
child_a = node.a
child_b = node.b

Call the child nodes and evaluate them.

In [4]:
result = child_a() + child_b()
result

30

### 2.2. Accessing Nested Fields

Define a nested dictionary structure.

In [5]:
node = SourceDeepTrackNode(lambda: {"a": 10, "b": {"c": 3, "d": 7}})

Access nested fields:

In [6]:
child_c = node.b.c
child_c()

3

In [7]:
child_d = node.b.d
child_d()

7

### 2.3. Integrating with a DeepTrack2 Feature

You can pass a `SourceDeepTrackNode` (or any field from a `Source`) to a DeepTrack2 feature such as `dt.Value`. When the data is activated (e.g., `source[i]()`), the feature dynamically evaluates the current value at that index.

Create a Source with two fields.

In [8]:
import deeptrack as dt
from deeptrack.sources import Source

source = Source(
    a=[1, 2, 3],
    b=[10, 20, 30],
)

Define a feature that sums fields `a` and `b`.

In [9]:
feature = dt.Value(source.a) + dt.Value(source.b)

Evaluate feature on each item in the source.

In [10]:
feature(source[0])

11

In [11]:
feature(source[1])

22

In [12]:
feature(source[2])

33

## 3. Using a `SourceItem` with Callbacks

A `SourceItem` is a dictionary-like object used internally by `Source`. It holds data fields (e.g., `a`, `b`) and triggers registered callbacks when the item is called.

This behavior enables features like automatic index switching and dependency tracking when a data item is accessed in DeepTrack2.

Define a callback function:

In [13]:
def callback(item):
    print(f"CALLBACK - Item accessed: {item}")

Create a `SourceItem` registering this callback:

In [14]:
from deeptrack.sources.base import SourceItem

item = SourceItem([callback], a=5, b=10)

Trigger the callback by calling the item:

In [15]:
item();

CALLBACK - Item accessed: SourceItem({'a': 5, 'b': 10}, 1 callback(s))


Access values directly as in a dictionary:

In [16]:
item["a"]

5

In [17]:
item["b"]

10

## 4. Using `Source` to Generate a Dataset of `SourceItem` Objects

A `Source` organizes multiple named sequences (e.g., `a`, `b`) into a dataset that returns SourceItem objects. These items can be accessed by index or iterated over.

This is the primary interface for constructing datasets in DeepTrack2.

### 4.1. Basic Usage

Define a source with multiple fields:

In [18]:
from deeptrack.sources.base import Source

dataset = Source(a=[1, 2, 3], b=[4, 5, 6])

Access individual items by index:

In [19]:
dataset[0]

SourceItem({'a': 1, 'b': 4}, 1 callback(s))

In [20]:
dataset[1]

SourceItem({'a': 2, 'b': 5}, 1 callback(s))

In [21]:
dataset[2]

SourceItem({'a': 3, 'b': 6}, 1 callback(s))

Iterate over all items in the source:

In [22]:
for item in dataset:
    print(item)

SourceItem({'a': 1, 'b': 4}, 1 callback(s))
SourceItem({'a': 2, 'b': 5}, 1 callback(s))
SourceItem({'a': 3, 'b': 6}, 1 callback(s))


### 4.2. Dynamically Evaluating Source Attributes

Each field in a `Source` is wrapped in a `SourceDeepTrackNode`, enabling dynamic evaluation based on the currently active index.

In [23]:
source = Source(a=[1, 2, 3], b=[10, 20, 30])
source.set_index(1)

Source(a=[1, 2, 3], b=[10, 20, 30])

In [24]:
source.a()

2

In [25]:
source.b()

20

Source attributes can be passed directly to features and dynamically evaluated using `SourceItem`.

In [26]:
feature = dt.Value(source.a) + dt.Value(source.b)
feature(source[2])

33

### 4.3. Registering Activation Callbacks with `on_activate()`

You can register callbacks that are triggered when a `SourceItem` is called. This enables features like dependency tracking, logging, or side effects.

Define a callback function:

In [27]:
def log_access(item):
    print(f"Item activated: {item}")

Define a source and add the callback:

In [28]:

source = Source(a=[1, 2], b=[3, 4])
source.on_activate(log_access)

Trigger the callback whern the source is called:

In [29]:
source[1]();

Item activated: SourceItem({'a': 2, 'b': 4}, 2 callback(s))


### 4.4. Slicing a Source

You can retrieve a contiguous subset of a `Source` using slicing. Each item in the result is a `SourceItem`.

Define a source:

In [30]:
source = Source(a=[1, 2, 3], b=[4, 5, 6])

Slice it:

In [31]:
subset = source[1:3]

Iterate over the sliced source items:

In [32]:
for item in subset:
    print(item)

SourceItem({'a': 2, 'b': 5}, 1 callback(s))
SourceItem({'a': 3, 'b': 6}, 1 callback(s))


### 4.5. Filtering a Source with `filter()`

The `filter()` method creates a `Subset` based on a predicate function. The predicate receives the fields as keyword arguments.

Create a source:

In [33]:
source = Source(
    a=[1, 2, 3, 4, 5, 6, 7, 8, 9],
    b=[11, 12, 13, 14, 15, 16, 17, 18, 19],
)

Filter it:

In [34]:
filtered = source.filter(lambda a, b: a > 2 and b < 16)

And iterate over the filtered items:

In [35]:
for item in filtered:
    print(item)

SourceItem({'a': 3, 'b': 13}, 1 callback(s))
SourceItem({'a': 4, 'b': 14}, 1 callback(s))
SourceItem({'a': 5, 'b': 15}, 1 callback(s))


### 4.6. Adding Constant Metadata with `constants()`

The `constants()` method adds new fields with fixed values to each item in the source. These are repeated to match the dataset length.

Create a source:

In [36]:
source = Source(a=[1, 2], b=[3, 4])

Add a constant:

In [37]:
labeled_source = source.constants(label="constant")
labeled_source

Product(label=['constant', 'constant'], a=[1, 2], b=[3, 4])

### 4.7.Expanding a Source with `product()`

The `product()` method creates a Cartesian product between the original items and new fields. This is useful for parameter sweeps or augmentation.

Create a source:

In [38]:
source = Source(a=[1, 2])

Extend the souce with a product:

In [39]:
extended = source.product(c=["x", "y"])

And iterate over the resulting source items:

In [40]:
for item in extended:
    print(item)

SourceItem({'c': 'x', 'a': 1}, 1 callback(s))
SourceItem({'c': 'y', 'a': 1}, 1 callback(s))
SourceItem({'c': 'x', 'a': 2}, 1 callback(s))
SourceItem({'c': 'y', 'a': 2}, 1 callback(s))


## 5. Combining Existing Attributes with `Product`

Create a source:

In [41]:
source = Source(a=[1, 2], b=[3, 4])

Generate a new source as a product with new attributes.

In [42]:
new_source = source.product(c=[5, 6])

Print the resulting combinations.

In [43]:
for item in new_source:
    print(item)

SourceItem({'c': 5, 'a': 1, 'b': 3}, 1 callback(s))
SourceItem({'c': 6, 'a': 1, 'b': 3}, 1 callback(s))
SourceItem({'c': 5, 'a': 2, 'b': 4}, 1 callback(s))
SourceItem({'c': 6, 'a': 2, 'b': 4}, 1 callback(s))


## 6. Filtering Dataset Items with `Subset`

Define a source:

In [44]:
source = Source(
    a=[1, 2, 3, 4, 2, 8, 8],
    b=[1, 2, 3, 4, 7, 9, 11],
)

Create a subset with only even values of `a`:

In [45]:
subset = source.filter(lambda a, b: a % 2 == 0)

Print the subset values:

In [46]:
for item in subset:
    print(item)

SourceItem({'a': 2, 'b': 2}, 1 callback(s))
SourceItem({'a': 4, 'b': 4}, 1 callback(s))
SourceItem({'a': 2, 'b': 7}, 1 callback(s))
SourceItem({'a': 8, 'b': 9}, 1 callback(s))
SourceItem({'a': 8, 'b': 11}, 1 callback(s))


## 7. Randomly Splitting Sources into Multiple Subsets

Create a source:

In [47]:
from deeptrack.sources import random_split
import numpy as np

source = Source(
    a=[1, 2, 3, 4, 5, 6],
    b=[7, 8, 9, 10, 11, 12],
)

Split into two subsets (proportionally to 70% and 30%) commonly used for validation during training.

In [48]:
train_subset, test_subset = random_split(
    source,
    lengths=[0.7, 0.3],
    generator=np.random.default_rng(42),
)

In [49]:
train_subset

Subset(a=[4, 3, 6, 5]..., b=[10, 9, 12, 11]...)

In [50]:
test_subset

Subset(a=[1], b=[7])